# StarGANv2-VC Demo (VCTK 20 Speakers)

### Utils

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Github/Accent-Conversion/
!git status




Mounted at /content/drive
/content/drive/MyDrive/Github/Accent-Conversion
Refresh index: 100% (34/34), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Configs/config.yml
	modified:   Data/VCTK.ipynb
	modified:   Demo/inference.ipynb
	modified:   project_notebook.ipynb
	modified:   train.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	Data download.ipynb
	Data/Data.zip
	Data/Processed_Data/
	Data/VCTK-Corpus-0.92.zip
	Data/p225/
	Data/p226/
	Data/p227/
	Data/p228/
	Data/p229/
	Data/p230/
	Data/p231/
	Data/p232/
	Data/p233/
	Data/p236/
	Data/p239/
	Data/p240/
	Data/p243/
	Data/p244/
	Data/p254/
	Data/p256/
	Data/p258/
	Data/p259/
	Data/p270/
	Data/p273/
	Data/speaker-info.txt
	Data/txt/
	Data/update.txt
	Data/wav48_silence_trimmed/
	DataSet_L2/
	Data_girl/

In [2]:
#%cd ..

!pip install -q munch
!pip install -q Utils
!pip install -q parallel_wavegan
!pip install -q scipy

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.3 MB/s eta 0:00:00


In [3]:


# load packages
import random
import yaml
from munch import Munch
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torchaudio
import librosa

from Utils.ASR.models import ASRCNN
from Utils.JDC.model import JDCNet
from models import Generator, MappingNetwork, StyleEncoder

%matplotlib inline

In [13]:
# Source: http://speech.ee.ntu.edu.tw/~jjery2243542/resource/model/is18/en_speaker_used.txt
# Source: https://github.com/jjery2243542/voice_conversion

speakers = ['ABA', 'SKA',    # Arabic [cite: 52]
    'BWC', 'LXC',    # Chinese [cite: 52]
    'ASI', 'SVBI',   # Hindi [cite: 52, 55]
    'HKK', 'HJK',    # Korean
    'EBVS', 'NJS',   # Spanish
    'HQTV', 'PNV'    # Vietnamese
]

to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300)
mean, std = -4, 4

def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor

def build_model(model_params={}):
    args = Munch(model_params)
    generator = Generator(args.dim_in, args.style_dim, args.max_conv_dim, w_hpf=args.w_hpf, F0_channel=args.F0_channel)
    mapping_network = MappingNetwork(args.latent_dim, args.style_dim, args.num_domains, hidden_dim=args.max_conv_dim)
    style_encoder = StyleEncoder(args.dim_in, args.style_dim, args.num_domains, args.max_conv_dim)

    nets_ema = Munch(generator=generator,
                     mapping_network=mapping_network,
                     style_encoder=style_encoder)

    return nets_ema

def compute_style(speaker_dicts):
    reference_embeddings = {}

    for key, (path, speaker) in speaker_dicts.items():
        if path == "":
            label = torch.LongTensor([speaker]).to(device)
            latent_dim = starganv2.mapping_network.shared[0].in_features
            z = torch.randn(1, latent_dim).to(device)
            ref = starganv2.mapping_network(z, label)
        else:
            wave, sr = librosa.load(path, sr=24000)
            wave, _ = librosa.effects.trim(wave, top_db=30)

            if sr != 24000:
                wave = librosa.resample(wave, orig_sr=sr, target_sr=24000)

            wave = wave / (np.max(np.abs(wave)) + 1e-9)
            mel_tensor = preprocess(wave).to(device)

            with torch.no_grad():
                label = torch.LongTensor([speaker]).to(device)
                ref = starganv2.style_encoder(mel_tensor.unsqueeze(1), label)

        reference_embeddings[key] = (ref, label)

    return reference_embeddings

### Load models

In [14]:
# load F0 model

device = torch.device("cpu")

F0_model = JDCNet(num_class=1, seq_len=192)
params = torch.load("Utils/JDC/bst.t7", map_location=device)['net']
F0_model.load_state_dict(params)
F0_model.eval()
F0_model = F0_model.to(device)

In [15]:
import scipy.signal
from scipy.signal.windows import kaiser
scipy.signal.kaiser = kaiser

# load vocoder
from parallel_wavegan.utils import load_model
vocoder = load_model("Vocoder/checkpoint-400000steps.pkl").to('cpu').eval()
vocoder.remove_weight_norm()
_ = vocoder.eval()

In [16]:
# load starganv2

model_path = 'Models/Non-Mother-Tone/epoch_00148.pth'

with open('Models/config.yml') as f:
    starganv2_config = yaml.safe_load(f)
starganv2 = build_model(model_params=starganv2_config["model_params"])
params = torch.load(model_path, map_location='cpu')
params = params['model_ema']
_ = [starganv2[key].load_state_dict(params[key]) for key in starganv2]
_ = [starganv2[key].eval() for key in starganv2]
starganv2.style_encoder = starganv2.style_encoder.to('cpu')
starganv2.mapping_network = starganv2.mapping_network.to('cpu')
starganv2.generator = starganv2.generator.to('cpu')

### Recording

In [17]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import os
import subprocess

record_dir = "Demo/live_inputs"
os.makedirs(record_dir, exist_ok=True)

def record_audio(filename="live_input.wav", sample_rate=24000):
    js = Javascript("""
    async function recordAudio() {
      const div = document.createElement('div');
      const startButton = document.createElement('button');
      const stopButton = document.createElement('button');

      startButton.textContent = 'Start Recording';
      stopButton.textContent = 'Stop Recording';

      div.appendChild(startButton);
      div.appendChild(stopButton);
      document.body.appendChild(div);

      let stream = await navigator.mediaDevices.getUserMedia({audio: true});
      let recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);

      const recorded = new Promise(resolve => {
        recorder.onstop = async () => {
          let blob = new Blob(chunks, {type: 'audio/webm'});
          let arrayBuffer = await blob.arrayBuffer();
          let uint8Array = new Uint8Array(arrayBuffer);
          let binary = '';
          uint8Array.forEach(b => binary += String.fromCharCode(b));
          resolve(btoa(binary));
        };
      });

      startButton.onclick = () => recorder.start();
      stopButton.onclick = () => recorder.stop();

      return await recorded;
    }
    """)
    display(js)
    audio_data = output.eval_js("recordAudio()")
    audio_bytes = b64decode(audio_data)

    webm_path = os.path.join(record_dir, "temp_record.webm")
    wav_path = os.path.join(record_dir, filename)

    with open(webm_path, "wb") as f:
        f.write(audio_bytes)

    subprocess.run([
        "ffmpeg", "-y",
        "-i", webm_path,
        "-ar", str(sample_rate),
        "-ac", "1",
        wav_path
    ], check=True)

    return wav_path

### Conversion

In [18]:
def load_recorded_source(wav_path):
    audio, source_sr = librosa.load(wav_path, sr=24000)
    audio = audio / (np.max(np.abs(audio)) + 1e-9)
    audio = audio.astype(np.float32)
    return audio

#### Convert by style encoder

In [29]:
# with reference, using style encoder
selected_speakers = ['BWC', 'LXC']

speaker_dicts = {}
for s in selected_speakers:
    ref_path = f"Data/Processed_Data/{s}/1.wav"
    speaker_dicts[f"p{s}"] = (ref_path, speakers.index(s))

reference_embeddings = compute_style(speaker_dicts)

In [30]:
import soundfile as sf
import IPython.display as ipd
import os
import librosa
import numpy as np
import torch

save_dir = "Demo/converted_results"
os.makedirs(save_dir, exist_ok=True)

def record_and_convert(filename="live_input.wav"):
    # 1. record
    live_wav_path = record_audio(filename=filename, sample_rate=24000)
    print("=" * 70)
    print("Recorded file:", live_wav_path)

    # 2. load source audio
    audio = load_recorded_source(live_wav_path)

    print("\nSource audio:")
    display(ipd.Audio(audio, rate=24000))

    # 3. source mel
    source = preprocess(audio).to(device)
    print("Source mel:")
    show_mel(source, title="Source Mel")

    # 4. convert to each reference speaker
    for ref_key, (ref, _) in reference_embeddings.items():
        print("\n" + "-" * 50)
        print(f"Reference speaker: {ref_key}")
        print(f"Reference path: {speaker_dicts[ref_key][0]}")

        # load reference audio
        ref_wave, _ = librosa.load(speaker_dicts[ref_key][0], sr=24000)
        ref_wave = ref_wave / (np.max(np.abs(ref_wave)) + 1e-9)

        print("Reference audio:")
        display(ipd.Audio(ref_wave, rate=24000))

        # reference mel
        ref_mel = preprocess(ref_wave).to(device)
        print("Reference mel:")
        show_mel(ref_mel, title=f"Reference Mel - {ref_key}")

        # conversion
        with torch.no_grad():
            f0_feat = F0_model.get_feature_GAN(source.unsqueeze(1))
            out = starganv2.generator(source.unsqueeze(1), ref, F0=f0_feat)

            print("Converted mel:")
            show_mel(out, title=f"Converted Mel - {ref_key}")

            c = out.transpose(-1, -2).squeeze().to(device)
            y_out = vocoder.inference(c)
            y_out = y_out.view(-1).cpu().numpy()

        # save converted result
        out_path = os.path.join(
            save_dir,
            f"{os.path.splitext(filename)[0]}_to_{ref_key}.wav"
        )
        sf.write(out_path, y_out, 24000)

        print("Converted audio:")
        display(ipd.Audio(y_out, rate=24000))
        print("Saved result:", out_path)

### Mel spectrogram plot

In [33]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def tensor_to_mel_numpy(mel_tensor):
    """
    Convert mel tensor to numpy with shape [n_mels, T]
    """
    if isinstance(mel_tensor, torch.Tensor):
        mel = mel_tensor.detach().cpu()

        if mel.dim() == 4:
            # [B, C, n_mels, T] -> [n_mels, T]
            mel = mel.squeeze(0).squeeze(0)
        elif mel.dim() == 3:
            # [C, n_mels, T] or [B, n_mels, T]
            mel = mel.squeeze(0)

        mel = mel.numpy()
    else:
        mel = mel_tensor

    # make sure shape is [n_mels, T]
    if mel.shape[0] > mel.shape[1]:
        # usually mel should be [80, T], so if not, transpose
        pass

    return mel

def show_mel(mel_tensor, title="Mel Spectrogram"):
    mel = tensor_to_mel_numpy(mel_tensor)

    plt.figure(figsize=(10, 4))
    plt.imshow(mel, aspect='auto', origin='lower')
    plt.title(title)
    plt.xlabel("Frames")
    plt.ylabel("Mel bins")
    plt.colorbar()
    plt.tight_layout()
    plt.show()

### Record and Convert

In [36]:
record_and_convert("my_test.wav")

Output hidden; open in https://colab.research.google.com to view.

In [37]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import rbf_kernel

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.primitives import Sampler

# =========================
# INPUT DATA
# =========================
# Replace with your dataset variables if different

X = X_train
y = y_train

# =========================
# PARAMETERS
# =========================

gamma = 1.0
reps = 2

# =========================
# SORT DATA BY LABEL
# =========================

sorted_indices = np.argsort(y)

X_sorted = X[sorted_indices]
y_sorted = y[sorted_indices]

# =========================
# CLASSICAL RBF KERNEL
# =========================

K_classical = rbf_kernel(
    X_sorted,
    X_sorted,
    gamma=gamma
)

# =========================
# QUANTUM KERNEL
# =========================

feature_map = ZZFeatureMap(
    feature_dimension=X.shape[1],
    reps=reps,
    entanglement="linear"
)

sampler = Sampler()

quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map,
    sampler=sampler
)

K_quantum = quantum_kernel.evaluate(
    x_vec=X_sorted
)

# =========================
# FIND CLASS BOUNDARIES
# =========================

unique_classes, counts = np.unique(y_sorted, return_counts=True)
boundaries = np.cumsum(counts)

# =========================
# PLOT HEATMAP
# =========================

plt.figure(figsize=(14,6))

# Classical kernel plot
plt.subplot(1,2,1)

sns.heatmap(
    K_classical,
    cmap="viridis",
    square=True,
    cbar=True
)

for boundary in boundaries[:-1]:
    plt.axhline(boundary, color='white', linewidth=1.5)
    plt.axvline(boundary, color='white', linewidth=1.5)

plt.title("Classical RBF Kernel Matrix", fontsize=14)


# Quantum kernel plot
plt.subplot(1,2,2)

sns.heatmap(
    K_quantum,
    cmap="viridis",
    square=True,
    cbar=True
)

for boundary in boundaries[:-1]:
    plt.axhline(boundary, color='white', linewidth=1.5)
    plt.axvline(boundary, color='white', linewidth=1.5)

plt.title("Quantum Kernel Matrix (ZZFeatureMap)", fontsize=14)


plt.tight_layout()

# =========================
# SAVE HIGH-QUALITY FIGURE
# =========================

plt.savefig(
    "kernel_matrix_compare.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved figure: kernel_matrix_compare.png")

ModuleNotFoundError: No module named 'qiskit'